## Generating AI charts

This notebook documents the process of generating AI-authored DDR step charts using the Dance Dance Convolution (DDC) model.

### Overview

The DDC model (Donahue et al., 2017) automatically generates step charts from audio files. It uses:
- A **C-LSTM model** for step placement (deciding when to place steps)
- A **conditional LSTM** for step selection (deciding which arrows to use)

### Source data
- **Input**: `data/unprocessed/raw/{fraxtil,itg}/**/*.ogg` (audio files)
- **Output**: `data/artificial/{fraxtil,itg}/*_ai.json`

### Requirements
The DDC model runs as a Docker container:
```bash
docker run -it -p 8080:80 chrisdonahue/ddc:latest
```


In [1]:
import json
import os
import re
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'unprocessed' / 'raw'
OUTPUT_DIR = DATA_DIR / 'artificial'
DDC_SERVER_URL = os.environ.get("DDC_SERVER_URL", "http://localhost:8080")
DIFFICULTIES = ['Beginner', 'Easy', 'Medium', 'Hard', 'Challenge']

In [2]:
def check_already_processed():
    existing = list(OUTPUT_DIR.glob('**/*_ai.json'))
    if existing:
        sample = existing[0]
        with open(sample, 'r') as f:
            data = json.load(f)
        if data.get('charts'):
            return True, len(existing)
    return False, 0

already_done, count = check_already_processed()
print(f"Already processed: {already_done} ({count} AI chart files found)")

Already processed: True (222 AI chart files found)


In [3]:
def parse_sm_notes(notes_text: str) -> list[dict]:
    steps = []
    measures = notes_text.strip().split(',')
    beat = 0.0
    
    for measure in measures:
        lines = [l.strip() for l in measure.strip().split('\n') 
                 if l.strip() and not l.strip().startswith('//')]
        if not lines:
            continue
        
        subdivision = len(lines)
        beats_per_line = 4.0 / subdivision
        
        for line in lines:
            if line != '0000':
                steps.append({'beat': beat, 'step': line})
            beat += beats_per_line
    
    return steps


def parse_sm_file(sm_content: str) -> dict:
    result = {'title': None, 'artist': None, 'bpm': None, 'charts': []}
    
    title_match = re.search(r'#TITLE:([^;]*);', sm_content)
    if title_match:
        result['title'] = title_match.group(1).strip()
    
    bpm_match = re.search(r'#BPMS:[\d.]*=([\d.]+);', sm_content)
    if bpm_match:
        result['bpm'] = float(bpm_match.group(1))
    
    notes_pattern = r'#NOTES:\s*([^:]*):([^:]*):([^:]*):([^:]*):([^:]*):([^;]*);'
    for match in re.finditer(notes_pattern, sm_content, re.DOTALL):
        chart_type = match.group(1).strip()
        difficulty = match.group(3).strip()
        difficulty_num = match.group(4).strip()
        notes_text = match.group(6)
        
        if chart_type == 'dance-single':
            steps = parse_sm_notes(notes_text)
            result['charts'].append({
                'difficulty': difficulty,
                'difficulty_num': int(difficulty_num) if difficulty_num.isdigit() else 0,
                'steps': steps,
                'num_steps': len(steps)
            })
    
    return result

In [4]:
def find_audio_files() -> list[tuple[Path, str, str]]:
    results = []
    
    for dataset in ['fraxtil', 'itg']:
        dataset_dir = RAW_DIR / dataset
        if not dataset_dir.exists():
            continue
        
        for pack_dir in dataset_dir.iterdir():
            if not pack_dir.is_dir():
                continue
            
            for song_dir in pack_dir.iterdir():
                if not song_dir.is_dir():
                    continue
                
                for audio_ext in ['.ogg', '.mp3', '.wav']:
                    audio_files = list(song_dir.glob(f'*{audio_ext}'))
                    if audio_files:
                        results.append((audio_files[0], dataset, song_dir.name))
                        break
    
    return results

if not already_done:
    audio_files = find_audio_files()
    print(f"Found {len(audio_files)} audio files")
else:
    print("Skipping audio file search - charts already generated")

Skipping audio file search - charts already generated


In [5]:
if already_done:
    print(f"Skipping: {count} AI chart files already exist in {OUTPUT_DIR}")
    print("\nTo regenerate, delete the existing files and rerun.")
else:
    print("""To generate AI charts, run the DDC Docker container and execute:
    
    docker run -it -p 8080:80 chrisdonahue/ddc:latest
    uv run python scripts/generate-ai-charts.py
    
The generation process:
1. Sends each audio file to the DDC /choreograph endpoint
2. Receives a .zip containing a .sm (StepMania) file
3. Parses the .sm file to extract step data
4. Saves the result as JSON
""")

Skipping: 222 AI chart files already exist in /Users/enscribe/Repositories/School/cse158-a2/data/artificial

To regenerate, delete the existing files and rerun.


In [6]:
sample_files = list(OUTPUT_DIR.glob('**/*_ai.json'))[:3]
for f in sample_files:
    with open(f, 'r') as file:
        data = json.load(file)
    print(f"\n{f.name}:")
    print(f"  Song: {data.get('song_name', 'Unknown')}")
    print(f"  Dataset: {data.get('dataset', 'Unknown')}")
    print(f"  Difficulties: {list(data.get('charts', {}).keys())}")
    for diff, chart in data.get('charts', {}).items():
        num_steps = chart.get('num_steps', len(chart.get('steps', [])))
        print(f"    {diff}: {num_steps} steps")


Zodiac_ai.json:
  Song: Zodiac
  Dataset: itg
  Difficulties: ['Beginner', 'Easy', 'Medium', 'Hard', 'Challenge']
    Beginner: 83 steps
    Easy: 244 steps
    Medium: 410 steps
    Hard: 592 steps
    Challenge: 703 steps

Go 60 Go_ai.json:
  Song: Go 60 Go
  Dataset: itg
  Difficulties: ['Beginner', 'Easy', 'Medium', 'Hard', 'Challenge']
    Beginner: 213 steps
    Easy: 416 steps
    Medium: 479 steps
    Hard: 573 steps
    Challenge: 834 steps

Hardcore Symphony_ai.json:
  Song: Hardcore Symphony
  Dataset: itg
  Difficulties: ['Beginner', 'Easy', 'Medium', 'Hard', 'Challenge']
    Beginner: 92 steps
    Easy: 289 steps
    Medium: 429 steps
    Hard: 529 steps
    Challenge: 618 steps
